# Count Expected FHIR Tool Calls per Task

This notebook analyzes FHIR task files to determine the **realistic minimum number of tool calls** an AI agent would need.

**Approach:**
- Sends entire task file (including `get_prompt()` and `execute_human_agent()`) to GPT-4
- Considers what information the agent has access to via the prompt
- If the agent needs IDs not mentioned in the prompt, counts additional search calls
- Outputs 0 for tasks with loops or unclear execution paths (for manual review)


In [9]:
import os
import re
import pandas as pd
from openai import OpenAI
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


In [10]:
# Path to task files
TASKS_DIR = Path("../tasks/fhir_tasks_modular")
task_files = sorted(TASKS_DIR.glob("task_*_modular.py"))


In [40]:
def read_full_task_file(file_path):
    """Read the entire task file."""
    with open(file_path, 'r') as f:
        return f.read()


In [41]:
def count_fhir_calls_with_llm(task_file_content, task_name):
    """Use GPT-4 to count realistic FHIR tool calls an agent would need.
    
    Returns:
        int: Number of FHIR calls, or 0 if there are loops or unclear logic
    """
    prompt = f"""You are analyzing FHIR healthcare agent tasks to determine how many tool calls an agent would need.

CONTEXT:
- These are Python task classes that test AI agents working with FHIR (Fast Healthcare Interoperability Resources) servers
- Each task has a `get_prompt()` method that defines what information the agent receives
- Each task has an `execute_human_agent()` method showing the ground-truth solution
- Agents have access to these FHIR tools:
  * getAllResources (search/list resources like Patients, Appointments, Slots)
  * getResourceById (fetch a specific resource by ID)
  * createResource (create new resources)
  * updateResource (update existing resources)
  * deleteResource (delete resources)
  * searchResources (search with parameters)

YOUR TASK:
Count the MINIMUM number of tool calls a realistic agent would need to complete the task.

CRITICAL RULES:
1. The agent ONLY knows what's in `get_prompt()` - if the prompt mentions "patient John Doe" but execute_human_agent uses the patient ID, the agent must SEARCH for it first
2. If `execute_human_agent()` uses hardcoded IDs (like "PAT001", "SLOT001") that aren't in the prompt, take that into consideration (e.g. count +1 search call to find them)
3. If there are loops (for/while), try to infer what the required number or necessary tool calls by looking at prepare_test_data.
4. Count the realistic agent's calls, not just what's in `execute_human_agent()`
5. if you can't determine the number of tool calls, return 0 (needs manual review)

Rules:
. If the execution path is unclear or conditional or if it contains loops, return 0
. Only return a single integer (the count)

Task file to analyze:
```python
{task_file_content}
```

Return ONLY a single integer representing the minimum tool calls needed. No explanation."""
    
    try:
        response = client.chat.completions.create(
            model="o3",
            messages=[
                {"role": "system", "content": "You are a code analyzer specialized in AI agent workflows. Return only integers."},
                {"role": "user", "content": prompt}
            ]
        )
        
        count_str = response.choices[0].message.content.strip()
        return int(count_str)
    except Exception as e:
        print(f"Error analyzing {task_name}: {e}")
        return 0


In [42]:
# Analyze all tasks
results = []

for task_file in task_files:
    task_name = task_file.stem  # e.g., 'task_01_enter_new_patient_modular'
    
    # Extract task ID from filename
    task_id_match = re.search(r'task_(\w+)_', task_name)
    task_id = task_id_match.group(1) if task_id_match else task_name
    
    print(f"Processing {task_name}...")
    
    # Read full task file
    task_content = read_full_task_file(task_file)
    
    if task_content is None:
        print(f"  ⚠️  Could not read task file")
        results.append({
            'task_id': task_id,
            'task_file': task_name,
            'expected_calls': -1,
            'note': 'File not found'
        })
        continue
    
    # Count calls using LLM
    call_count = count_fhir_calls_with_llm(task_content, task_name)
    
    note = ''
    if call_count == 0:
        note = 'Loop/unclear - needs manual review'
    
    results.append({
        'task_id': task_id,
        'task_file': task_name,
        'expected_calls': call_count,
        'note': note
    })
    
    print(f"  ✓ Expected calls: {call_count}")

print(f"\n✅ Processed {len(results)} tasks")


Processing task_01_enter_new_patient_modular...
  ✓ Expected calls: 1
Processing task_02a_search_existing_patient_modular...
  ✓ Expected calls: 1
Processing task_02b_search_nonexistent_patient_modular...
  ✓ Expected calls: 1
Processing task_03_enter_medical_history_modular...
  ✓ Expected calls: 1
Processing task_04a_search_nonempty_medical_history_modular...
  ✓ Expected calls: 1
Processing task_04b_search_empty_medical_history_modular...
  ✓ Expected calls: 1
Processing task_05_enter_surgery_plan_modular...
  ✓ Expected calls: 1
Processing task_06a_search_existing_surgery_plan_modular...
  ✓ Expected calls: 1
Processing task_06b_search_nonexistent_surgery_plan_modular...
  ✓ Expected calls: 1
Processing task_07_enter_insurance_modular...
  ✓ Expected calls: 3
Processing task_08a_search_existing_insurance_modular...
  ✓ Expected calls: 1
Processing task_08b_search_nonexistent_insurance_modular...
  ✓ Expected calls: 1
Processing task_09a_create_related_person_modular...
  ✓ Expected

In [30]:
results

[{'task_id': '01_enter_new_patient',
  'task_file': 'task_01_enter_new_patient_modular',
  'expected_calls': 1,
  'note': ''},
 {'task_id': '02a_search_existing_patient',
  'task_file': 'task_02a_search_existing_patient_modular',
  'expected_calls': 1,
  'note': ''},
 {'task_id': '02b_search_nonexistent_patient',
  'task_file': 'task_02b_search_nonexistent_patient_modular',
  'expected_calls': 1,
  'note': ''},
 {'task_id': '03_enter_medical_history',
  'task_file': 'task_03_enter_medical_history_modular',
  'expected_calls': 1,
  'note': ''},
 {'task_id': '04a_search_nonempty_medical_history',
  'task_file': 'task_04a_search_nonempty_medical_history_modular',
  'expected_calls': 1,
  'note': ''},
 {'task_id': '04b_search_empty_medical_history',
  'task_file': 'task_04b_search_empty_medical_history_modular',
  'expected_calls': 1,
  'note': ''},
 {'task_id': '05_enter_surgery_plan',
  'task_file': 'task_05_enter_surgery_plan_modular',
  'expected_calls': 1,
  'note': ''},
 {'task_id': 

In [31]:
results_o3 = results

In [33]:
# Create DataFrame and display results
df = pd.DataFrame(results_4_1_mini)
df = pd.DataFrame(results_4_1)
# df = pd.DataFrame(results_o3)

df


,task_id,task_file,expected_calls,note
0,01_enter_new_patient,task_01_enter_new_patient_modular,1,
1,02a_search_existing_patient,task_02a_search_existing_patient_modular,1,
2,02b_search_nonexistent_patient,task_02b_search_nonexistent_patient_modular,1,
3,03_enter_medical_history,task_03_enter_medical_history_modular,1,
4,04a_search_nonempty_medical_history,task_04a_search_nonempty_medical_history_modular,1,
5,04b_search_empty_medical_history,task_04b_search_empty_medical_history_modular,1,
6,05_enter_surgery_plan,task_05_enter_surgery_plan_modular,1,
7,06a_search_existing_surgery_plan,task_06a_search_existing_surgery_plan_modular,1,
8,06b_search_nonexistent_surgery_plan,task_06b_search_nonexistent_surgery_plan_modular,1,
9,07_enter_insurance,task_07_enter_insurance_modular,3,


In [ ]:
# Show tasks that need manual review (expected_calls == 0)
needs_review = df[df['expected_calls'] == 0]
print(f"\n🔍 Tasks needing manual review: {len(needs_review)}\n")
needs_review


In [ ]:
# Summary statistics
print("Summary Statistics:")
print(f"Total tasks: {len(df)}")
print(f"Tasks with clear counts: {len(df[df['expected_calls'] > 0])}")
print(f"Tasks needing review: {len(df[df['expected_calls'] == 0])}")
print(f"\nCall count distribution:")
print(df[df['expected_calls'] > 0]['expected_calls'].value_counts().sort_index())


In [44]:
# Save simplified results to CSV
# Create simplified dataframe with just task_id and expected_calls
df_simple = pd.DataFrame({
    'task': df['task_id'],
    'expected_tool_calls': df['expected_calls']
})
df_simple

,task,expected_tool_calls
0,01_enter_new_patient,1
1,02a_search_existing_patient,1
2,02b_search_nonexistent_patient,1
3,03_enter_medical_history,1
4,04a_search_nonempty_medical_history,1
5,04b_search_empty_medical_history,1
6,05_enter_surgery_plan,1
7,06a_search_existing_surgery_plan,1
8,06b_search_nonexistent_surgery_plan,1
9,07_enter_insurance,3


In [45]:

output_file = "data/expected_tool_calls_per_task.csv"
df_simple.to_csv(output_file, index=False)
print(f"\n💾 Simplified results saved to: {output_file}")
print(f"Format: task,expected_tool_calls")
print(f"Example rows:")
print(df_simple.head())



💾 Simplified results saved to: data/expected_tool_calls_per_task.csv
Format: task,expected_tool_calls
Example rows:
                                  task  expected_tool_calls
0                 01_enter_new_patient                    1
1          02a_search_existing_patient                    1
2       02b_search_nonexistent_patient                    1
3             03_enter_medical_history                    1
4  04a_search_nonempty_medical_history                    1


In [4]:
import pandas as pd
df = pd.read_csv("data/expected_tool_calls_per_task.csv")

In [8]:
df['expected_tool_calls'].value_counts()

expected_tool_calls
1    21
0    18
2     4
3     3
6     1
7     1
4     1
Name: count, dtype: int64